# ACE-Step 1.5 server for Raaga.AI (demo, base model)

This notebook runs **ACE-Step `acestep-v15-base`** on a Colab GPU and exposes it through a public HTTPS tunnel. Your Raaga.AI app then works like this:
1. Your **svara transformer** (`data/models/svara_transformer.pt`) composes the raga-correct melody and the Telugu lyrics.
2. Raaga.AI renders a guide track from them.
3. This server re-performs the guide with realistic vocals and instruments.
4. The app keeps the most raga-faithful candidate.

Runtime → Change runtime type → **T4 GPU** (L4 or A100 is faster). The base model runs 50 diffusion steps, so expect a few minutes per song on a T4.

**Keep this tab open.** The server stops when the Colab session ends, and the tunnel URL changes each time you start it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/raagaai'
os.makedirs(f'{DRIVE_DIR}/ace_checkpoints', exist_ok=True)
# Model weights persist on Drive, so only the first run downloads them (~10 GB)
os.environ['ACESTEP_CHECKPOINTS_DIR'] = f'{DRIVE_DIR}/ace_checkpoints'

In [ ]:
%cd /content
!test -d ACE-Step-1.5 || git clone -q --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
!pip install -q -e /content/ACE-Step-1.5
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Start the server
The first start downloads the models to Drive, which can take 10–20 minutes.

In [ ]:
import secrets, subprocess, time, urllib.request

ACESTEP_MODEL = 'acestep-v15-base'   # matches ACESTEP_MODEL in Raaga.AI's .env
API_KEY = secrets.token_urlsafe(24)

env = dict(os.environ,
           ACESTEP_CONFIG_PATH=ACESTEP_MODEL,
           ACESTEP_INIT_LLM='false')       # cover mode never uses the 5Hz LM
server = subprocess.Popen(
    ['acestep-api', '--host', '0.0.0.0', '--port', '8001', '--api-key', API_KEY],
    cwd='/content/ACE-Step-1.5', env=env,
    stdout=open('/content/acestep.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(240):                        # wait up to 40 min for downloads + load
    try:
        if urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=5).status == 200:
            print('ACE-Step server is up'); break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError('server exited — see the log cell below')
    time.sleep(10)
else:
    raise TimeoutError('server did not come up — see the log cell below')

In [ ]:
!tail -n 30 /content/acestep.log

## 2. Open a public tunnel

In [ ]:
import re
!test -f /content/cloudflared || wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8001'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0); break
    time.sleep(2)
assert url, 'no tunnel URL — check /content/tunnel.log'

print('Put these in Raaga.AI\'s .env, then restart the API / worker:\n')
print(f'ACESTEP_API_URL={url}')
print(f'ACESTEP_API_KEY={API_KEY}')
print(f'ACESTEP_MODEL={ACESTEP_MODEL}')
print('ACESTEP_INFERENCE_STEPS=50')

## 3. Optional: test a full song here
This runs the same pipeline as the app: your svara transformer if `data/models/svara_transformer.pt` exists (otherwise the rule-based fallback), then the guide, then ACE-Step and ranking.
To use your trained model, copy it from Drive into `raagaai/data/models/` first.

In [ ]:
%cd /content
!test -d raagaai || git clone -q -b feat/hf-training-hybrid-song https://github.com/Pariseashishbannu/raagaai.git
!pip install -q numpy scipy soundfile structlog pydantic-settings librosa httpx
# Use your trained model from train_colab.ipynb if it's on Drive
!mkdir -p raagaai/data/models && cp -n {DRIVE_DIR}/finetune/svara_transformer.pt {DRIVE_DIR}/finetune/tokenizer_vocab.json raagaai/data/models/ 2>/dev/null; ls raagaai/data/models

os.environ.update(ACESTEP_API_URL='http://127.0.0.1:8001', ACESTEP_API_KEY=API_KEY,
                  ACESTEP_MODEL=ACESTEP_MODEL, ACESTEP_INFERENCE_STEPS='50')
%cd /content/raagaai
!python -m scripts.hybrid_song --raga mohanam --theme devotional --voice female --out-dir /content/song

In [ ]:
import json
from IPython.display import Audio, display
meta = json.load(open('/content/song/song.json'))
print(meta['song_title'], '| performed by', meta['performed_by'], '| raga score', meta.get('raga_score'))
print(meta['sahityam'])
display(Audio('/content/song/guide.wav' if meta['performed_by'] == 'ace-step' else '/content/song/composition.wav'))
display(Audio('/content/song/composition.wav'))